# VietHandOCR Part 1: Data Preparation & EDA

Welcome to **Part 1** of the VietHandOCR pipeline. 
- **Next Notebook**: [Part 2: Baseline Evaluation](./02_Baseline_Evaluation.ipynb)

## Introduction
This notebook handles extracting the UIT-HWDB dataset, optimizing memory (downcasting), and performing a strict writer-independent split. We save the intermediate results to `.txt` files for the next notebook.


In [1]:
import os, gc, zipfile, random
import numpy as np
import pandas as pd

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)
print("Seeded everything successfully!")


Seeded everything successfully!


In [2]:
def reduce_mem_usage(df):
    '''Iterates through columns and modifies data types to reduce memory usage.'''
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and not pd.api.types.is_categorical_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max: df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
                else: df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max: df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
                else: df[col] = df[col].astype(np.float64)
        else:
            num_unique_values = len(df[col].unique())
            num_total_values = len(df[col])
            if num_unique_values / num_total_values < 0.5:
                df[col] = df[col].astype('category')
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'\n[2/4] TỐI ƯU HÓA BỘ NHỚ (MEMORY REDUCTION)...')
    print(f'  - Dung lượng ban đầu: {start_mem:.2f} MB')
    print(f'  - Dung lượng sau tối ưu: {end_mem:.2f} MB (Giảm {((start_mem - end_mem) / start_mem * 100):.1f}%)')
    return df


In [3]:
import csv
def writer_independent_split(metadata_df, val_size=0.1):
    '''Splits data ensuring writers in validation set are not in training set.'''
    if 'writer_id' not in metadata_df.columns:
        from sklearn.model_selection import train_test_split
        return train_test_split(metadata_df, test_size=val_size, random_state=42)
        
    writers = list(metadata_df['writer_id'].unique())
    np.random.shuffle(writers)
    split_idx = int(len(writers) * (1 - val_size))
    train_writers, val_writers = writers[:split_idx], writers[split_idx:]
    
    train_df = metadata_df[metadata_df['writer_id'].isin(train_writers)].copy()
    val_df = metadata_df[metadata_df['writer_id'].isin(val_writers)].copy()
    
    print(f'\n[3/4] THỰC HIỆN CHIA TÁCH (DATA SPLIT)...')
    print(f'  - Tìm thấy {len(writers)} tác giả trong tập Train/Val.')
    print(f'  - Phân bổ {len(train_writers)} tác giả ({100*(1-val_size):.0f}%) vào tập Train ({len(train_df)} ảnh).')
    print(f'  - Phân bổ {len(val_writers)} tác giả ({100*val_size:.0f}%) vào tập Validation ({len(val_df)} ảnh) để đảm bảo không trùng lặp tác giả.')
    return train_df, val_df

import json
import glob
import os

def build_metadata(base_dir):
    '''Crawls the UIT-HWDB structure to build a pandas DataFrame from label.json files.'''
    print(f'\n[1/4] BẮT ĐẦU QUÁ TRÌNH TÌM KIẾM DỮ LIỆU...')
    print(f'  - Dataset Input Path: {base_dir}')
    
    records = []
    json_paths = glob.glob(os.path.join(base_dir, '**', 'label.json'), recursive=True)
    print(f'  - Đã quét được {len(json_paths)} file label.json.')
    
    for json_path in json_paths:
        parts = json_path.split(os.sep)
        if len(parts) >= 4:
            level = parts[-4]
            split = parts[-3]
            writer_id = parts[-2]
            
            with open(json_path, 'r', encoding='utf-8') as f:
                try:
                    labels = json.load(f)
                    for img_name, text in labels.items():
                        img_path = os.path.join(os.path.dirname(json_path), img_name)
                        records.append({
                            'image_path': img_path[img_path.find('UIT_HWDB_'):] if 'UIT_HWDB_' in img_path else img_path,
                            'label': text,
                            'writer_id': writer_id,
                            'level': level,
                            'split': split
                        })
                except json.JSONDecodeError:
                    print(f"  [LỖI] Không thể đọc file: {json_path}")
                    
    df = pd.DataFrame(records)
    print(f'  - Trích xuất thành công {len(df)} dòng dữ liệu.')
    if len(df) > 0 and 'level' in df.columns:
        for lvl, count in df['level'].value_counts().items():
            print(f'    + Cấp độ {lvl}: {count} ảnh')
    return df

base_dataset_path = next((os.path.join('/kaggle/input', d) for d in os.listdir('/kaggle/input') if os.path.isdir(os.path.join('/kaggle/input', d))), 'VietHandOCR_Datasets') if os.path.exists('/kaggle/input') else 'VietHandOCR_Datasets'
metadata = build_metadata(base_dataset_path)
metadata = reduce_mem_usage(metadata)

train_val_data = metadata[metadata['split'] == 'train_data']
test_data = metadata[metadata['split'] == 'test_data']

print(f'  - Tách tập test_data độc lập: {len(test_data)} ảnh.')

train_df, val_df = writer_independent_split(train_val_data)

print(f"\n[4/4] EXPORTING DATASET SPLITS")
print(f"{'='*50}")
print(f"Global Splits (All levels combined):")
print(f"  - Train size: {len(train_df)}")
print(f"  - Val size:   {len(val_df)}")
print(f"  - Test size:  {len(test_data)}\n")

# Save global splits
train_df[['image_path', 'label']].to_csv('train_all.txt', sep='\t', index=False, header=False, quoting=csv.QUOTE_NONE, escapechar='\\')
val_df[['image_path', 'label']].to_csv('val_all.txt', sep='\t', index=False, header=False, quoting=csv.QUOTE_NONE, escapechar='\\')
test_data[['image_path', 'label']].to_csv('test_all.txt', sep='\t', index=False, header=False, quoting=csv.QUOTE_NONE, escapechar='\\')
print("✅ Saved global splits: train_all.txt, val_all.txt, test_all.txt")

# Define mapping for clean filenames
level_mapping = {
    'UIT_HWDB_word': 'word',
    'UIT_HWDB_line': 'line',
    'UIT_HWDB_paragraph': 'paragraph'
}

print(f"\nLevel-specific Splits:")
for level in train_df['level'].unique():
    suffix = level_mapping.get(level, level.split('_')[-1])
    
    # Filter datasets
    t_df = train_df[train_df['level'] == level]
    v_df = val_df[val_df['level'] == level]
    test_df = test_data[test_data['level'] == level]
    
    # Output to files
    t_df[['image_path', 'label']].to_csv(f'train_{suffix}.txt', sep='\t', index=False, header=False, quoting=csv.QUOTE_NONE, escapechar='\\')
    v_df[['image_path', 'label']].to_csv(f'val_{suffix}.txt', sep='\t', index=False, header=False, quoting=csv.QUOTE_NONE, escapechar='\\')
    test_df[['image_path', 'label']].to_csv(f'test_{suffix}.txt', sep='\t', index=False, header=False, quoting=csv.QUOTE_NONE, escapechar='\\')
    
    print(f"  - Level '{suffix}' -> train_{suffix}.txt ({len(t_df)}), val_{suffix}.txt ({len(v_df)}), test_{suffix}.txt ({len(test_df)})")

print(f"\n{'='*50}")
print("✅ DATA EXPORT COMPLETED SUCCESSFULLY!")
print(f"All 12 files have been written to: {os.getcwd()}")
print(f"{'='*50}")

# Cleanup
del metadata, train_val_data, train_df, val_df, test_data
import gc
gc.collect()



[1/4] BẮT ĐẦU QUÁ TRÌNH TÌM KIẾM DỮ LIỆU...
  - Dataset Input Path: /kaggle/input/datasets
  - Đã quét được 765 file label.json.
  - Trích xuất thành công 118861 dòng dữ liệu.
    + Cấp độ UIT_HWDB_word: 110488 ảnh
    + Cấp độ UIT_HWDB_line: 7229 ảnh
    + Cấp độ UIT_HWDB_paragraph: 1144 ảnh

[2/4] TỐI ƯU HÓA BỘ NHỚ (MEMORY REDUCTION)...
  - Dung lượng ban đầu: 4.53 MB
  - Dung lượng sau tối ưu: 1.93 MB (Giảm 57.5%)
  - Tách tập test_data độc lập: 3113 ảnh.

[3/4] THỰC HIỆN CHIA TÁCH (DATA SPLIT)...
  - Tìm thấy 249 tác giả trong tập Train/Val.
  - Phân bổ 224 tác giả (90%) vào tập Train (103724 ảnh).
  - Phân bổ 25 tác giả (10%) vào tập Validation (12024 ảnh) để đảm bảo không trùng lặp tác giả.

[4/4] EXPORTING DATASET SPLITS
Global Splits (All levels combined):
  - Train size: 103724
  - Val size:   12024
  - Test size:  3113

✅ Saved global splits: train_all.txt, val_all.txt, test_all.txt

Level-specific Splits:
  - Level 'paragraph' -> train_paragraph.txt (999), val_paragraph.txt

0